# ALDIMI - Merge salud y stock

Este notebook prepara el dataset de stock con la estructura requerida y lo vincula con el dataset de salud.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
DATA_DIR = Path('data')
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
MERGED_DIR = DATA_DIR / 'merged'

for d in (RAW_DIR, PROCESSED_DIR, MERGED_DIR):
    d.mkdir(parents=True, exist_ok=True)

health_df = pd.read_csv(RAW_DIR / 'health_raw.csv')
stock_raw = pd.read_csv(RAW_DIR / 'stock_raw.csv')

print('Health shape:', health_df.shape)
print('Stock shape:', stock_raw.shape)

Health shape: (2000, 21)
Stock shape: (91250, 15)


In [4]:
def find_col(df, keywords):
    for col in df.columns:
        low = col.lower()
        if any(k in low for k in keywords):
            return col
    return None

def ensure_fecha(df):
    df = df.copy()
    date_col = find_col(df, ['date', 'fecha', 'day'])
    if date_col:
        df['Fecha'] = pd.to_datetime(df[date_col], errors='coerce')
        df['Fecha'] = df['Fecha'].fillna(pd.Timestamp('2023-01-01'))
    else:
        df['Fecha'] = pd.date_range('2023-01-01', periods=len(df), freq='D')
    return df

In [5]:
# Salud: crear Alto_Riesgo y ocupacion total por fecha
health_df = ensure_fecha(health_df)
risk_col = find_col(health_df, ['risk_level', 'risk level', 'risklevel'])
score_col = find_col(health_df, ['risk_score', 'overall_risk', 'score'])

if risk_col:
    health_df['Alto_Riesgo'] = health_df[risk_col].astype(str).str.lower().str.contains('high')
elif score_col:
    threshold = health_df[score_col].median()
    health_df['Alto_Riesgo'] = pd.to_numeric(health_df[score_col], errors='coerce') >= threshold
else:
    health_df['Alto_Riesgo'] = False

health_daily = health_df.groupby('Fecha').agg(
    Pacientes_Alto_Riesgo=('Alto_Riesgo', 'sum'),
    Ocupacion_Total=('Alto_Riesgo', 'count')
).reset_index()

health_daily.to_csv(PROCESSED_DIR / 'health_daily.csv', index=False)
print('Saved health daily:', PROCESSED_DIR / 'health_daily.csv')
health_daily.head()

Saved health daily: data\processed\health_daily.csv


,Fecha,Pacientes_Alto_Riesgo,Ocupacion_Total
0,2023-01-01,0,1
1,2023-01-02,0,1
2,2023-01-03,0,1
3,2023-01-04,0,1
4,2023-01-05,0,1


In [7]:
# Stock: estructurar columnas requeridas
stock_df = ensure_fecha(stock_raw)
item_col = find_col(stock_df, ['item', 'sku', 'product', 'category', 'material', 'name'])
stock_level_col = find_col(stock_df, ['stock', 'inventory', 'onhand', 'on_hand', 'quantity', 'qty'])
lead_col = find_col(stock_df, ['lead', 'replenish', 'reorder', 'lt'])

if item_col:
    item_str = stock_df[item_col].astype(str).str.lower()
    stock_df['ID_Insumo'] = np.where(
        item_str.str.contains('drug|med|pharma|medicine'),
        'medicinas',
        'alimentos'
    )
else:
    stock_df['ID_Insumo'] = 'alimentos'

if stock_level_col:
    stock_df['Stock_Actual'] = pd.to_numeric(stock_df[stock_level_col], errors='coerce').fillna(0)
else:
    rng = np.random.default_rng(42)
    stock_df['Stock_Actual'] = rng.integers(50, 500, size=len(stock_df))

if lead_col:
    stock_df['Lead_Time'] = pd.to_numeric(stock_df[lead_col], errors='coerce').fillna(7).astype(int)
else:
    rng = np.random.default_rng(42)
    stock_df['Lead_Time'] = rng.integers(2, 15, size=len(stock_df))

stock_base = stock_df[['Fecha', 'ID_Insumo', 'Stock_Actual', 'Lead_Time']].copy()
stock_with_health = stock_base.merge(health_daily, on='Fecha', how='left')

stock_with_health['Pacientes_Alto_Riesgo'] = stock_with_health['Pacientes_Alto_Riesgo'].fillna(0)
stock_with_health['Ocupacion_Total'] = stock_with_health['Ocupacion_Total'].fillna(0)

rng = np.random.default_rng(42)
noise = rng.normal(0, 5, size=len(stock_with_health))
stock_with_health['Consumo_Diario'] = (
    20
    + 0.4 * stock_with_health['Pacientes_Alto_Riesgo']
    + 0.1 * stock_with_health['Ocupacion_Total']
    + noise
).clip(lower=1).round(2)

stock_with_health['Ocupacion_Albergue'] = (
    0.6 * stock_with_health['Ocupacion_Total']
    + rng.normal(0, 2, size=len(stock_with_health))
).clip(lower=0).round(2)

stock_structured = stock_with_health[[
    'Fecha',
    'ID_Insumo',
    'Stock_Actual',
    'Consumo_Diario',
    'Lead_Time',
    'Ocupacion_Albergue'
]].copy()

stock_structured.to_csv(PROCESSED_DIR / 'stock_structured.csv', index=False)
print('Saved stock structured:', PROCESSED_DIR / 'stock_structured.csv')
stock_structured.head()

Saved stock structured: data\processed\stock_structured.csv


,Fecha,ID_Insumo,Stock_Actual,Consumo_Diario,Lead_Time,Ocupacion_Albergue
0,2024-01-01,alimentos,592,21.62,14,0.68
1,2024-01-02,alimentos,575,14.90,14,0.00
2,2024-01-03,alimentos,540,23.85,14,0.00
3,2024-01-04,alimentos,516,25.20,14,1.69
4,2024-01-05,alimentos,495,10.34,14,0.00


In [8]:
# Merge final de salud y stock
merged = stock_structured.merge(health_daily, on='Fecha', how='left')
merged_out = MERGED_DIR / 'Dataset_ALDIMI_Merged.csv'
merged.to_csv(merged_out, index=False)
print('Saved merged dataset:', merged_out)

corr = merged[['Consumo_Diario', 'Pacientes_Alto_Riesgo', 'Ocupacion_Total']].corr()
print('Correlation check:')
print(corr)

merged.head()

Saved merged dataset: data\merged\Dataset_ALDIMI_Merged.csv
Correlation check:
                       Consumo_Diario  Pacientes_Alto_Riesgo  Ocupacion_Total
Consumo_Diario               1.000000               0.011057              NaN
Pacientes_Alto_Riesgo        0.011057               1.000000              NaN
Ocupacion_Total                   NaN                    NaN              NaN


,Fecha,ID_Insumo,Stock_Actual,Consumo_Diario,Lead_Time,Ocupacion_Albergue,Pacientes_Alto_Riesgo,Ocupacion_Total
0,2024-01-01,alimentos,592,21.62,14,0.68,0,1
1,2024-01-02,alimentos,575,14.90,14,0.00,0,1
2,2024-01-03,alimentos,540,23.85,14,0.00,0,1
3,2024-01-04,alimentos,516,25.20,14,1.69,1,1
4,2024-01-05,alimentos,495,10.34,14,0.00,0,1
